# 🚀 GIAI ĐOẠN 6 — KIẾN TRÚC STAIR-ICAART (STAIR-PEEC & STAIR-REEC)
### POOLED & REPLICATED EXCESS-EVIDENCE CALIBRATION TRÊN ĐỒ THỊ NGỮ NGHĨA
**Thực hiện bởi:** Nhóm KLTN Recommender Systems (Lê Hà Thanh Chương & Bùi Trung Hiếu)  
**Đề tài:** Recommender Systems using Graph Representation: Multi-modal  
**Nền tảng:** STAIR5-v8 / WMSG-core (Không gian 64 chiều, tối ưu hóa qua Item-side BSC)  
**Quy chuẩn thực nghiệm:** Đối chuẩn đa thế hệ (Baseline, v4, v8, v8.1 BSCV), 0 tham số bổ sung, bảo toàn chuẩn toán tử $\|S_{\text{ICA}}\|_2 \le 1.0$.

---
*Notebook này được tổ chức theo chuẩn thực nghiệm của Khóa luận tốt nghiệp (tương tự STAIR6-v1, v2): các cell training tách biệt theo từng pha, có vẽ biểu đồ Learning Curve và VRAM Profile độc lập cho từng tập dữ liệu, bảng đối chuẩn đa thế hệ và kiểm định thống kê.*


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR-ICAART

In [ ]:
# Cell 1: Môi trường, Dependencies & Xác thực Subprocess STAIR-ICAART
import os, shutil, subprocess, sys, glob, json, hashlib
from pathlib import Path

print('=' * 80)
print('🚀 THIẾT LẬP MÔI TRƯỜNG & ĐỒNG BỘ MÃ NGUỒN STAIR-ICAART (versionICAART)...')
print('=' * 80)

_safe_home = os.path.expanduser('~')
_kw = '/kaggle/working'
os.makedirs(_kw, exist_ok=True)

# 1. Tìm vị trí STAIR-Enhanced hiện hữu
_possible_stair_dirs = [
    os.path.join(_kw, 'STAIR-Enhanced'),
    os.path.join(_safe_home, 'STAIR-Enhanced'),
    os.path.abspath('STAIR-Enhanced'),
    os.path.abspath('.'),
]
_stair_dir = None
for _p in _possible_stair_dirs:
    if os.path.isdir(_p) and (os.path.exists(os.path.join(_p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(_p, 'main_stair_icaart.py'))):
        _stair_dir = _p
        break

if _stair_dir is None:
    _stair_dir = os.path.join(_kw, 'STAIR-Enhanced') if os.path.exists(_kw) else os.path.abspath('STAIR-Enhanced')
    if not os.path.exists(_stair_dir):
        print('📦 Đang clone repository ThanhChuong12/STAIR-Enhanced (branch main)...')
        try:
            subprocess.run(
                ['git', 'clone', '--depth', '1', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', _stair_dir],
                check=True
            )
            print('✅ Clone thành công!')
        except Exception as _e:
            print(f'⚠️ Git clone gặp lỗi: {_e}')
    else:
        print('Thư mục STAIR-Enhanced đã tồn tại. Đang đồng bộ cập nhật git pull mới nhất...')
        try:
            subprocess.run(['git', '-C', _stair_dir, 'pull', 'origin', 'main'], check=False, capture_output=True)
            print('✅ Đã kéo cập nhật mới nhất từ origin/main.')
        except Exception:
            pass

# Fallback: Tự động dò quét và sao chép mã nguồn từ Kaggle Input nếu chưa có models/stair_icaart.py
if not os.path.exists(os.path.join(_stair_dir, 'models', 'stair_icaart.py')):
    _input_matches = glob.glob('/kaggle/input/**/models/stair_icaart.py', recursive=True)
    if _input_matches:
        _src_repo = os.path.dirname(os.path.dirname(_input_matches[0]))
        print(f'📦 Phát hiện mã nguồn STAIR-ICAART từ Kaggle Input: {_src_repo} -> Sao chép vào {_stair_dir}...')
        os.makedirs(_stair_dir, exist_ok=True)
        shutil.copytree(_src_repo, _stair_dir, dirs_exist_ok=True)

# Tạo liên kết symlink 2 chiều giữa ~/STAIR-Enhanced và /kaggle/working/STAIR-Enhanced
_home_stair = os.path.join(_safe_home, 'STAIR-Enhanced')
_kw_stair = os.path.join(_kw, 'STAIR-Enhanced')
if os.path.exists(_home_stair) and not os.path.exists(_kw_stair):
    try:
        os.symlink(_home_stair, _kw_stair)
    except Exception:
        pass
elif os.path.exists(_kw_stair) and not os.path.exists(_home_stair):
    try:
        os.symlink(_kw_stair, _home_stair)
    except Exception:
        pass

# Tạo symlink các thư mục modules cốt lõi vào /kaggle/working để pytest, CLI và import tìm thấy trực tiếp
for _sub in ['models', 'configs', 'tests', 'tools', 'main_stair_icaart.py', 'stair_icaart_config.py', 'stair_icaart_reporting.py']:
    _src_sub = os.path.join(_stair_dir, _sub)
    _dst_sub = os.path.join(_kw, _sub)
    if os.path.exists(_src_sub) and not os.path.exists(_dst_sub) and not os.path.islink(_dst_sub):
        try:
            os.symlink(_src_sub, _dst_sub)
        except Exception:
            pass

# Đưa vào sys.path
for _p in [_stair_dir, _kw, _safe_home]:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# Thiết lập biến toàn cục chuẩn mực cho toàn bộ Notebook
REPO_ROOT = Path(_stair_dir).resolve()
WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
STAIR_DIR = REPO_ROOT

print(f'✅ Thư mục mã nguồn REPO_ROOT: {REPO_ROOT}')
print(f'✅ Thư mục làm việc WORK_DIR: {WORK_DIR}')

# 2. Cài đặt các gói phụ thuộc tiêu chuẩn với cơ chế chống lỗi mạng/xung đột
def _install_deps():
    _pkgs = ['prettytable', 'scipy', 'matplotlib', 'seaborn', 'pandas', 'pytest', 'nvidia-ml-py']
    for _pkg in _pkgs:
        try:
            __import__(_pkg)
        except ImportError:
            print(f'📦 Đang cài đặt {_pkg}...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)

    try:
        import freerec
    except ImportError:
        print('📦 Đang cài đặt FreeRec...')
        _r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'freerec==0.9.7'], check=False)
        if _r.returncode != 0:
            print('⚠️ Cài đặt freerec tiêu chuẩn gặp lỗi, thử chế độ --no-deps...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'freerec'], check=False)

    try:
        import torch_geometric
    except ImportError:
        print('📦 Đang cài đặt torch_geometric...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torch_geometric'], check=False)

    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=False)

_install_deps()

# 3. Cấu hình biến môi trường Subprocess
ENV = os.environ.copy()
ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
ENV['PYTHONUTF8'] = '1'
ENV['PYTHONIOENCODING'] = 'utf-8'
ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Đảm bảo các thư mục output sẵn sàng
for _d in [str(WORK_DIR / 'logs' / 'stair_icaart'), str(WORK_DIR / 'reports'), str(WORK_DIR / 'runs' / 'stair_icaart' / 'graph_cache')]:
    os.makedirs(_d, exist_ok=True)
    try:
        os.makedirs(os.path.expanduser(f'~/{os.path.basename(_d)}'), exist_ok=True)
    except Exception:
        pass

# 4. Kiểm tra tính tương thích phần cứng & module STAIR-ICAART
_verify_code = """
import sys, torch
import models.freerec_compat
import freerec
from models.stair_icaart import STAIR_ICAART_Model
from stair_icaart_config import validate_icaart_config, DEFAULTS
from models.stair_icaart_evidence import (
    build_activity_replication, replicated_excess_score, compute_replicated_excess_evidence,
    build_activity_strata, pooled_activity_score
)
from models.stair_icaart_graph import calibrate_semantic_support

print(f"🐍 Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | FreeRec: {freerec.__version__}")
print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   ↳ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)")
print("✅ Khế ước import module STAIR-ICAART (PEEC & REEC) đã được kiểm tra thành công.")
"""
_res = subprocess.run([sys.executable, '-c', _verify_code], env=ENV, cwd=str(REPO_ROOT), capture_output=True, text=True)
print(_res.stdout)
if _res.stderr and 'warning' not in _res.stderr.lower():
    print('⚠️ Cảnh báo:', _res.stderr)
print('=' * 80)


## Cell 2 📂 Chuẩn bị Dữ liệu (KaggleHub Auto-Fetch & Multi-Bridge sang /kaggle/data & Processed)

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input / KaggleHub sang /kaggle/data & Processed
import os, shutil, glob
from pathlib import Path

_DATA_ROOT = '/kaggle/data' if os.path.exists('/kaggle') else str(REPO_ROOT / 'data')
_PROCESSED_ROOT = os.path.join(_DATA_ROOT, 'Processed')
_LOCAL_DATA = str(REPO_ROOT / 'data')
_LOCAL_PROCESSED = os.path.join(_LOCAL_DATA, 'Processed')

for _d in [_DATA_ROOT, _PROCESSED_ROOT, _LOCAL_DATA, _LOCAL_PROCESSED]:
    os.makedirs(_d, exist_ok=True)

_TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

_REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')

def _bridge_directories(src_dir, target_folder):
    """Bắc cầu thư mục dữ liệu bằng cách tạo thư mục thực writable và symlink từng file để tránh lỗi Read-only filesystem."""
    _destinations = [
        os.path.join(_DATA_ROOT, target_folder),
        os.path.join(_PROCESSED_ROOT, target_folder),
        os.path.join(_LOCAL_DATA, target_folder),
        os.path.join(_LOCAL_PROCESSED, target_folder),
    ]
    for _dst in _destinations:
        if os.path.abspath(src_dir) == os.path.abspath(_dst):
            continue
        if os.path.islink(_dst):
            try:
                os.unlink(_dst)
            except Exception:
                pass
        os.makedirs(_dst, exist_ok=True)

        _chunks_dir = os.path.join(_dst, 'chunks')
        if os.path.islink(_chunks_dir):
            try:
                os.unlink(_chunks_dir)
            except Exception:
                pass
        os.makedirs(_chunks_dir, exist_ok=True)

        for _item in os.listdir(src_dir):
            if _item == 'chunks':
                continue
            _s_item = os.path.join(src_dir, _item)
            _d_item = os.path.join(_dst, _item)
            if os.path.isdir(_s_item):
                if os.path.islink(_d_item):
                    try:
                        os.unlink(_d_item)
                    except Exception:
                        pass
                os.makedirs(_d_item, exist_ok=True)
                for _sub_f in os.listdir(_s_item):
                    _s_sub = os.path.join(_s_item, _sub_f)
                    _d_sub = os.path.join(_d_item, _sub_f)
                    if not os.path.exists(_d_sub):
                        try:
                            os.symlink(_s_sub, _d_sub)
                        except Exception:
                            shutil.copy2(_s_sub, _d_sub)
            else:
                if not os.path.exists(_d_item) or os.path.getsize(_d_item) == 0:
                    try:
                        os.symlink(_s_item, _d_item)
                    except Exception:
                        shutil.copy2(_s_item, _d_item)

def _find_existing_dataset(folder_name, aliases):
    _search_bases = ['/kaggle/input', '/kaggle/data', str(REPO_ROOT / 'data'), '.', '..', str(WORK_DIR), os.path.expanduser('~')]
    for _base in _search_bases:
        if not os.path.exists(_base):
            continue
        for _root, _dirs, _files in os.walk(_base):
            _bname = os.path.basename(_root).lower()
            if _bname == folder_name.lower() or any(_alias in _bname for _alias in aliases):
                if all(_req in _files for _req in _REQUIRED_FILES):
                    return _root
    return None

print('=' * 80)
print('📂 TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR-REEC...')
print('=' * 80)

_missing = []
for _key, (_folder, _aliases) in _TARGET_DATASETS.items():
    if not _find_existing_dataset(_folder, _aliases):
        _missing.append(_key)

if _missing:
    print(f'📦 Phát hiện thiếu {_missing}. Đang kích hoạt KaggleHub auto-download...')
    try:
        import kagglehub
        _slug = 'rainyle/stair-datasets-mmrec'
        print(f'  ↳ Đang tải dataset {_slug} qua kagglehub...')
        _dl_path = kagglehub.dataset_download(_slug)
        print(f'  ↳ Tải thành công tại: {_dl_path}')

        if os.path.exists('/kaggle'):
            os.makedirs('/kaggle/input', exist_ok=True)
            _in_symlink = '/kaggle/input/stair-datasets-mmrec'
            if not os.path.exists(_in_symlink) and not os.path.islink(_in_symlink):
                try:
                    os.symlink(_dl_path, _in_symlink)
                except Exception:
                    pass
    except Exception as _e:
        print(f'⚠️ KaggleHub download error: {_e}')

for _key, (_folder, _aliases) in _TARGET_DATASETS.items():
    _found = _find_existing_dataset(_folder, _aliases)
    if _found:
        _bridge_directories(_found, _folder)
        print(f'  ✅ Đã bắc cầu thành công tập dữ liệu [{_key.upper()}]: {_found} -> {_DATA_ROOT}/{_folder}')
    else:
        print(f'  ❌ Chưa tìm thấy tập dữ liệu: {_folder}')

print('=' * 80)


## Cell 3 🧪 Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR-ICAART (40 Unit & Integration Tests)

In [ ]:
# Cell 3: Chạy bộ kiểm thử độc lập STAIR-ICAART (40 Unit & Integration Tests)
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

import subprocess, sys

print('=' * 80)
print('🧪 KHỞI CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR-ICAART (PEEC & REEC)...')
print('=' * 80)

_test_files = [
    'tests/test_stair_peec_evidence.py',
    'tests/test_stair_icaart_reporting.py',
    'tests/test_stair_icaart_graph.py',
    'tests/test_stair_icaart_pipeline.py'
]
_resolved_tests = [str(REPO_ROOT / _f) for _f in _test_files if (REPO_ROOT / _f).exists()]

if not _resolved_tests:
    print('⚠️ Cảnh báo: Không tìm thấy tệp test trong REPO_ROOT/tests. Đang kiểm tra /kaggle/working/tests...')
    _resolved_tests = [str(WORK_DIR / _f) for _f in _test_files if (WORK_DIR / _f).exists()]

_test_cmd = [sys.executable, '-m', 'pytest', '-v', '-W', 'ignore::DeprecationWarning', '-W', 'ignore::UserWarning'] + _resolved_tests
print('Command:', ' '.join(_test_cmd))
_res_test = subprocess.run(_test_cmd, capture_output=True, text=True, env=ENV, cwd=str(REPO_ROOT))
print(_res_test.stdout)
if _res_test.stderr and 'warning' not in _res_test.stderr.lower():
    print('⚠️ Test stderr:', _res_test.stderr)

if _res_test.returncode == 0:
    print('✅ TẤT CẢ 40 BÀI KIỂM THỬ TOÁN HỌC & ĐỒ THỊ STAIR-ICAART ĐỀU ĐẠT CHUẨN 100%!')
else:
    print(f'❌ Kiểm thử hoàn tất với mã thoát {_res_test.returncode}.')
print('=' * 80)


## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU Hardware Profiler & Visualization

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, json, re, threading, subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'

# Bảng số liệu đối soát tham chiếu đa thế hệ chính thức
TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

BASELINE_REF = {
    'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V4_REF = {
    'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
    'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
    'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
}

V6_REF = {
    'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517, 'epoch': 485},
    'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0677, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@1': 0.0099, 'Recall@10': 0.0458, 'Recall@20': 0.0681, 'NDCG@10': 0.0259, 'NDCG@20': 0.0317, 'epoch': 440},
}

V7_REF = {
    'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0768, 'Recall@20': 0.1147, 'NDCG@10': 0.0422, 'NDCG@20': 0.0520, 'epoch': 490},
    'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0684, 'NDCG@10': 0.0262, 'NDCG@20': 0.0318, 'epoch': 485},
}

V8_64_REF = {
    'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
    'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
    'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

STAIR6_V1_REF = {
    'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
    'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
    'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

STAIR6_V2_REF = {
    'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
    'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
    'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

vram_profile = {'sports': [], 'baby': [], 'electronics': []}
STAIR_ICAART_RESULTS = {}

def vram_monitor(key, stop_event, interval=2.0):
    nvml_ok = False
    handle = None
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        nvml_ok = True
    except Exception:
        pass

    while not stop_event.is_set():
        val = 0.0
        if nvml_ok and handle:
            try:
                info = pynvml.nvmlDeviceGetMemoryInfo(handle)
                val = info.used / (1024 ** 2)
            except Exception:
                pass
        elif torch.cuda.is_available():
            try:
                val = torch.cuda.max_memory_allocated() / (1024 ** 2)
            except Exception:
                pass
        if val > 0:
            vram_profile[key].append(val)
        time.sleep(interval)

    if nvml_ok:
        try:
            import pynvml
            pynvml.nvmlShutdown()
        except Exception:
            pass

def extract_test_metrics(log_path, artifact_dir=None):
    """Trích xuất chỉ số Test chính thức tại best model checkpoint."""
    metrics = {}
    selected_epoch = None
    if not os.path.exists(log_path):
        return selected_epoch, metrics

    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    for idx, line in enumerate(lines):
        load_m = re.search(r'Load best model @Epoch\s*:\s*(\d+)', line, re.I)
        if load_m:
            selected_epoch = int(load_m.group(1))

        test_m = re.search(r'\bTEST\s+@Epoch\s*:\s*(\d+)', line, re.I)
        if test_m and selected_epoch is not None and int(test_m.group(1)) == selected_epoch:
            for sub_line in lines[idx:idx+15]:
                for metric in TRACKED_METRICS:
                    pattern = re.escape(metric) + r'\s+Avg\s*:\s*([0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?)'
                    m = re.search(pattern, sub_line, re.I)
                    if m:
                        metrics[metric] = float(m.group(1))

    if artifact_dir and os.path.exists(artifact_dir):
        sm_file = os.path.join(artifact_dir, 'selected_test_metrics.json')
        if os.path.exists(sm_file):
            try:
                with open(sm_file, 'r', encoding='utf-8') as sf:
                    j_data = json.load(sf)
                    if 'test_metrics' in j_data:
                        for k, v in j_data['test_metrics'].items():
                            metrics[k] = float(v)
                    if 'selected_epoch' in j_data and selected_epoch is None:
                        selected_epoch = int(j_data['selected_epoch'])
            except Exception:
                pass

    return selected_epoch, metrics

def extract_best_validation(log_path):
    """Trích xuất best validation epoch và NDCG@20 cao nhất."""
    best_epoch = None
    best_val_ndcg = -1.0
    if not os.path.exists(log_path):
        return best_epoch, best_val_ndcg

    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            m = re.search(r'VALID\s+@Epoch\s*:\s*(\d+).*?NDCG@20\s+Avg\s*:\s*([0-9]*\.?[0-9]+)', line, re.I)
            if m:
                ep = int(m.group(1))
                val = float(m.group(2))
                if val > best_val_ndcg:
                    best_val_ndcg = val
                    best_epoch = ep
    return best_epoch, best_val_ndcg

def parse_telemetry_jsonl(artifact_dir):
    """Đọc chuỗi dữ liệu telemetry từ training_telemetry.jsonl."""
    t_file = os.path.join(artifact_dir, 'training_telemetry.jsonl')
    records = []
    if not os.path.exists(t_file):
        return records
    with open(t_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return records

def parse_training_losses(log_path, telemetry=None):
    """Trích xuất lịch sử BPR Loss và Total Loss."""
    bpr_losses = []
    cl_losses = []
    total_losses = []

    if telemetry:
        for r in telemetry:
            ep = r.get('epoch')
            if ep is not None:
                if 'train_loss' in r:
                    total_losses.append((ep, float(r['train_loss'])))
                if 'bpr_loss' in r:
                    bpr_losses.append((ep, float(r['bpr_loss'])))
                if 'cl_loss' in r:
                    cl_losses.append((ep, float(r['cl_loss'])))
        if total_losses:
            return bpr_losses, cl_losses, total_losses

    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                m = re.search(r'TRAIN\s+@Epoch\s*:\s*(\d+).*?LOSS\s+Avg\s*:\s*([0-9]*\.?[0-9]+)', line, re.I)
                if m:
                    ep = int(m.group(1))
                    val = float(m.group(2))
                    total_losses.append((ep, val))
    return bpr_losses, cl_losses, total_losses

def parse_valid_metric(log_path, metric_name='NDCG@20'):
    """Trích xuất tiến trình Validation metric qua các epoch."""
    vals = []
    if not os.path.exists(log_path):
        return vals
    pattern = re.compile(r'VALID\s+@Epoch\s*:\s*(\d+).*?' + re.escape(metric_name) + r'\s+Avg\s*:\s*([0-9]*\.?[0-9]+)', re.I)
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                vals.append((int(m.group(1)), float(m.group(2))))
    return vals

def parse_icaart_diagnostics(log_path):
    """Trích xuất bảng chẩn đoán cơ chế STAIR-REEC (versionICAART) từ log."""
    diag = {}
    if not os.path.exists(log_path):
        return diag
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()
    
    patterns = {
        'evidence_mode': r'Evidence Mode:\s*(.+)',
        'evidence_delta': r'Delta \(strength\):\s*([0-9.]+)',
        'evidence_tau': r'Tau \(count regularizer\):\s*([0-9.]+)',
        'exact_v8_bypass': r'Exact V8 Bypass:\s*(True|False)',
        'num_candidate_edges': r'Candidate Edges E:\s*([0-9]+)',
        'edges_with_excess': r'Edges with score > 0:\s*([0-9]+)\s*\(([0-9.]+)%\)',
        'weighted_coverage': r'Weighted Coverage:\s*([0-9.]+)%',
        'single_half_disagreement': r'Single-Half Disagreement:\s*([0-9.]+)%',
        'endpoint_contrast_mean': r'Endpoint Contrast.*?mean=([0-9.]+)',
        'relative_operator_dose': r'Operator Relative Dose:\s*([0-9.]+)',
        's_ica_fingerprint': r'S_ICA Fingerprint:\s*([a-f0-9]+)',
    }
    for k, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            diag[k] = m.group(1)
    return diag

def run_training_stair_icaart(key, config_yaml=None, data_root=None, log_path=None, artifact_dir=None,
                              delta=1.0, tau=2.0, mode='replicated_excess', preprocess_only=False, **kwargs):
    """Thực thi tiến trình huấn luyện STAIR-REEC (versionICAART) với real-time log streaming và VRAM monitoring."""
    cfg_dict = STAIR_ICAART_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)
    cfg_dict['evidence_delta'] = delta
    cfg_dict['evidence_tau'] = tau
    cfg_dict['evidence_mode'] = mode

    config_yaml = config_yaml or cfg_dict.get('yaml')
    label = f"{key}_STAIR_ICAART_{mode}_d{delta}_seed{cfg_dict.get('seed', 1)}"
    log_path = log_path or str(Path(cfg_dict['log']).parent / (label + '.log'))
    artifact_dir = artifact_dir or str(Path(cfg_dict['artifact_dir']).parent / label)
    resuming = bool(cfg_dict.get('resume_from'))

    if not resuming and (Path(log_path).exists() or Path(artifact_dir).exists()):
        print(f'⚠️ Khởi tạo phiên thử nghiệm sạch cho {label}.')
        if Path(artifact_dir).exists():
            for stale in ('manifest.json', 'training_checkpoint.pt', 'training_telemetry.jsonl', 'best_model.pt', 'last_model.pt', 'selected_test_metrics.json'):
                sp_file = Path(artifact_dir) / stale
                if sp_file.exists():
                    try:
                        sp_file.unlink()
                    except OSError:
                        pass

    STAIR_ICAART_CONFIGS[key].update(log=log_path, artifact_dir=artifact_dir)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    # Tìm file runner main_stair_icaart.py với cơ chế fallback thông minh
    _possible_runners = [
        str(REPO_ROOT / 'main_stair_icaart.py'),
        str(WORK_DIR / 'STAIR-Enhanced' / 'main_stair_icaart.py'),
        str(WORK_DIR / 'main_stair_icaart.py'),
        os.path.expanduser('~/STAIR-Enhanced/main_stair_icaart.py'),
        'main_stair_icaart.py',
    ]
    runner_py = next((r for r in _possible_runners if os.path.exists(r)), str(REPO_ROOT / 'main_stair_icaart.py'))

    # Tìm config yaml với cơ chế fallback thông minh
    if not os.path.exists(config_yaml):
        for cand in [
            str(REPO_ROOT / 'configs' / os.path.basename(config_yaml)),
            str(WORK_DIR / 'STAIR-Enhanced' / 'configs' / os.path.basename(config_yaml)),
            str(WORK_DIR / 'configs' / os.path.basename(config_yaml)),
            os.path.join('configs', os.path.basename(config_yaml)),
            os.path.expanduser(f'~/STAIR-Enhanced/configs/{os.path.basename(config_yaml)}'),
        ]:
            if os.path.exists(cand):
                config_yaml = cand
                break

    if data_root is None:
        data_root = '/kaggle/data' if os.path.exists('/kaggle/data') else (str(REPO_ROOT / 'data') if (REPO_ROOT / 'data').exists() else 'data')

    print('=' * 80)
    print(f'🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR-REEC: {key.upper()}')
    print(f'  * Dataset Key             : {key}')
    print(f'  * Config YAML             : {config_yaml}')
    print(f'  * Evidence Mode           : {mode}')
    print(f'  * Calibration Delta (δ)   : {delta}')
    print(f'  * Count Regularizer (τ)   : {tau}')
    print(f'  * Seed                    : {cfg_dict.get("seed", 1)}')
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [
        sys.executable, '-u', str(runner_py),
        '--config', str(config_yaml),
        '--root', str(data_root),
        '--device', str(device_str),
        '--ranking', 'full',
        '--artifact-dir', str(artifact_dir),
        '--evidence-mode', str(mode),
        '--evidence-delta', str(delta),
        '--evidence-tau', str(tau),
    ]

    for k_item in ('evidence_activity_bins', 'evidence_halves', 'evidence_seed', 'evidence_candidate_block',
                   'v8_arm', 'semantic_mode', 'edge_power', 'edge_floor', 'semantic_mix', 'alignment',
                   'relation_gate', 'relation_strength', 'bsc_residual', 'lambda_dirichlet', 'seed', 'eval_freq',
                   'eta', 'k_cf', 'c_min', 't_shrinkage', 'cf_block_size', 'lambda_nlgcl', 'nlgcl_tau',
                   'nlgcl_G', 'nlgcl_alpha', 'cl_chunk_size', 'knn_chunk_size', 'knn_device'):
        if k_item in cfg_dict:
            cmd += ['--' + k_item.replace('_', '-'), str(cfg_dict[k_item])]

    for k_item in ('epochs', 'batch_size', 'gamma', 'lr', 'weight_decay', 'num_workers'):
        if k_item in cfg_dict:
            cmd += ['--' + k_item.replace('_', '-'), str(cfg_dict[k_item])]

    if cfg_dict.get('resume_from'):
        cmd += ['--resume-from', str(cfg_dict['resume_from'])]
    if preprocess_only:
        cmd += ['--preprocess-only']

    print('Command:', ' '.join(cmd))
    _active_dir = os.path.dirname(os.path.abspath(runner_py))
    _cmd_env = ENV.copy()
    _cmd_env['PYTHONPATH'] = os.pathsep.join(filter(None, ['.', _active_dir, str(REPO_ROOT), str(WORK_DIR), os.environ.get('PYTHONPATH', '')]))

    try:
        with open(log_path, 'a' if resuming else 'w', encoding='utf-8') as lf:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8', errors='replace',
                bufsize=1,
                env=_cmd_env,
                cwd=_active_dir,
            )
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            proc.wait()
            ret_code = proc.returncode
    finally:
        stop_evt.set()
        th.join(timeout=3.0)

    elapsed = time.time() - t0
    print(f'⏱️ Thời gian thực thi ({key.upper()}): {elapsed/60:.2f} phút | Trạng thái exit code: {ret_code}')
    if ret_code != 0:
        raise RuntimeError(f'Tiến trình huấn luyện {key.upper()} thất bại với mã lỗi {ret_code}. Kiểm tra log: {log_path}')
    return elapsed


## Cell 5 📋 Cấu hình Siêu tham số STAIR-ICAART (PEEC & REEC Dataset-Adaptive Matrix)

In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR-ICAART (Dataset-Adaptive Matrix & Ablation Arms)
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

import time
from prettytable import PrettyTable

# TẠO RUN_ID ĐỘC NHẤT
if 'RUN_ID' not in globals():
    RUN_ID = time.strftime('%Y%m%d_%H%M%S') + f'_{os.urandom(3).hex()}'

# TUỲ CHỌN MÔ HÌNH: 'PEEC' (Primary R2 - Pooled Activity Excess) hoặc 'REEC' (Historical R1 - Replicated Excess)
ACTIVE_ESTIMATOR = 'PEEC'  # Chuyển thành 'REEC' nếu muốn chạy nhánh kiểm chứng chia nửa độc lập

def _resolve_yaml(filename):
    cands = [
        REPO_ROOT / 'configs' / filename,
        WORK_DIR / 'configs' / filename,
        Path(f'/kaggle/working/STAIR-Enhanced/configs/{filename}'),
        Path(f'/kaggle/working/configs/{filename}'),
    ]
    for c in cands:
        if c.exists():
            return str(c)
    return str(REPO_ROOT / 'configs' / filename)

if ACTIVE_ESTIMATOR == 'PEEC':
    _ev_mode = 'pooled_activity_excess'
    _ev_halves = 0
    _prefix = 'PEEC'
    _yaml_map = {
        'sports': _resolve_yaml('Amazon2014Sports_STAIR_PEEC.yaml'),
        'baby': _resolve_yaml('Amazon2014Baby_STAIR_PEEC.yaml'),
        'electronics': _resolve_yaml('Amazon2014Electronics_STAIR_PEEC.yaml'),
    }
else:
    _ev_mode = 'replicated_excess'
    _ev_halves = 2
    _prefix = 'REEC'
    _yaml_map = {
        'sports': _resolve_yaml('Amazon2014Sports_STAIR_ICAART.yaml'),
        'baby': _resolve_yaml('Amazon2014Baby_STAIR_ICAART.yaml'),
        'electronics': _resolve_yaml('Amazon2014Electronics_STAIR_ICAART.yaml'),
    }

STAIR_ICAART_CONFIGS = {
    'sports': {
        'dataset':                       'Amazon2014Sports_550_MMRec',
        'yaml':                          _yaml_map['sports'],
        'log':                           str(WORK_DIR / 'logs' / 'stair_icaart' / f'{RUN_ID}' / f'sports_STAIR_{_prefix}_d1.0_seed1.log'),
        'artifact_dir':                  str(WORK_DIR / 'runs' / 'stair_icaart' / f'{RUN_ID}' / f'sports_{_prefix}_seed1'),
        'evidence_mode':                 _ev_mode,
        'evidence_delta':                1.0,
        'evidence_tau':                  2.0,
        'evidence_activity_bins':        'log2',
        'evidence_halves':               _ev_halves,
        'evidence_seed':                 20261016,
        'evidence_candidate_block':      8192,
        'v8_arm':                        'WMSG-core',
        'semantic_mode':                 'weighted',
        'edge_power':                    2.0,
        'edge_floor':                    0.05,
        'semantic_mix':                  1.0,
        'alignment':                     'off',
        'relation_gate':                 'off',
        'eta':                           0.1,
        'k_cf':                          5,
        'c_min':                         2,
        't_shrinkage':                   5.0,
        'cf_block_size':                 64,
        'lambda_nlgcl':                  0.01,
        'nlgcl_tau':                     0.2,
        'nlgcl_G':                       1,
        'nlgcl_alpha':                   0.5,
        'cl_chunk_size':                 1024,
        'seed':                          1,
        'eval_freq':                     5,
    },
    'baby': {
        'dataset':                       'Amazon2014Baby_550_MMRec',
        'yaml':                          _yaml_map['baby'],
        'log':                           str(WORK_DIR / 'logs' / 'stair_icaart' / f'{RUN_ID}' / f'baby_STAIR_{_prefix}_d1.0_seed1.log'),
        'artifact_dir':                  str(WORK_DIR / 'runs' / 'stair_icaart' / f'{RUN_ID}' / f'baby_{_prefix}_seed1'),
        'evidence_mode':                 _ev_mode,
        'evidence_delta':                1.0,
        'evidence_tau':                  2.0,
        'evidence_activity_bins':        'log2',
        'evidence_halves':               _ev_halves,
        'evidence_seed':                 20261016,
        'evidence_candidate_block':      8192,
        'v8_arm':                        'WMSG-core',
        'semantic_mode':                 'weighted',
        'edge_power':                    2.0,
        'edge_floor':                    0.05,
        'semantic_mix':                  1.0,
        'alignment':                     'off',
        'relation_gate':                 'off',
        'eta':                           0.1,
        'k_cf':                          5,
        'c_min':                         2,
        't_shrinkage':                   5.0,
        'cf_block_size':                 64,
        'lambda_nlgcl':                  0.01,
        'nlgcl_tau':                     0.2,
        'nlgcl_G':                       1,
        'nlgcl_alpha':                   0.5,
        'cl_chunk_size':                 1024,
        'seed':                          1,
        'eval_freq':                     5,
    },
    'electronics': {
        'dataset':                       'Amazon2014Electronics_550_MMRec',
        'yaml':                          _yaml_map['electronics'],
        'log':                           str(WORK_DIR / 'logs' / 'stair_icaart' / f'{RUN_ID}' / f'electronics_STAIR_{_prefix}_d1.0_seed1.log'),
        'artifact_dir':                  str(WORK_DIR / 'runs' / 'stair_icaart' / f'{RUN_ID}' / f'electronics_{_prefix}_seed1'),
        'evidence_mode':                 _ev_mode,
        'evidence_delta':                1.0,
        'evidence_tau':                  2.0,
        'evidence_activity_bins':        'log2',
        'evidence_halves':               _ev_halves,
        'evidence_seed':                 20261016,
        'evidence_candidate_block':      8192,
        'v8_arm':                        'WMSG-core',
        'semantic_mode':                 'weighted',
        'edge_power':                    2.0,
        'edge_floor':                    0.05,
        'semantic_mix':                  1.0,
        'alignment':                     'off',
        'relation_gate':                 'off',
        'eta':                           0.1,
        'k_cf':                          5,
        'c_min':                         2,
        't_shrinkage':                   5.0,
        'cf_block_size':                 64,
        'lambda_nlgcl':                  0.01,
        'nlgcl_tau':                     0.2,
        'nlgcl_G':                       1,
        'nlgcl_alpha':                   0.5,
        'cl_chunk_size':                 128,
        'seed':                          1,
        'eval_freq':                     5,
    }
}

print('=' * 90)
print(f'📋 CẤU HÌNH SIÊU THAM SỐ STAIR-ICAART [{ACTIVE_ESTIMATOR}] (RUN_ID: {RUN_ID})')
print('=' * 90)
_t = PrettyTable()
_t.field_names = ['Tham số', 'Amazon Sports', 'Amazon Baby', 'Amazon Electronics']
_params_to_show = [
    ('Dataset', 'dataset'),
    ('Estimator Mode', 'evidence_mode'),
    ('Evidence Delta (δ)', 'evidence_delta'),
    ('Count Scale (τ)', 'evidence_tau'),
    ('User Halves (H)', 'evidence_halves'),
    ('Backbone Arm', 'v8_arm'),
    ('Semantic Mix (ε)', 'semantic_mix'),
    ('CF Weight (η)', 'eta'),
    ('NLGCL Weight (λ)', 'lambda_nlgcl'),
    ('CL Chunk Size', 'cl_chunk_size'),
]
for label, key in _params_to_show:
    _t.add_row([label, STAIR_ICAART_CONFIGS['sports'][key], STAIR_ICAART_CONFIGS['baby'][key], STAIR_ICAART_CONFIGS['electronics'][key]])
print(_t)
print('=' * 90)


## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot STAIR-ICAART)

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 6a: Training STAIR-REEC on Amazon Sports (Pha A: Pilot)
sports_time = run_training_stair_icaart('sports', delta=1.0, mode='replicated_excess')
ep_sports, metrics_sports = extract_test_metrics(STAIR_ICAART_CONFIGS['sports']['log'], STAIR_ICAART_CONFIGS['sports']['artifact_dir'])
val_ep_sports, val_m_sports = extract_best_validation(STAIR_ICAART_CONFIGS['sports']['log'])
diag_sports = parse_icaart_diagnostics(STAIR_ICAART_CONFIGS['sports']['log'])

STAIR_ICAART_RESULTS['sports'] = {
    'metrics': metrics_sports,
    'best_epoch': ep_sports,
    'val_metrics': val_m_sports,
    'train_time': sports_time,
    'diagnostics': diag_sports,
}

print('=' * 80)
print(f'🏆 KẾT QUẢ NGHIỆM THU AMAZON SPORTS (STAIR-REEC / Best Epoch: {ep_sports}):')
for m, v in metrics_sports.items():
    v8_val = V8_64_REF['sports'].get(m, 0.0)
    bl_val = BASELINE_REF['sports'].get(m, 0.0)
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    delta_bl = ((v - bl_val) / bl_val) * 100 if bl_val else 0.0
    print(f'  * {m:12s}: {v:.4f} (Baseline: {bl_val:.4f} | Δ vs BL: {delta_bl:+.2f}% | v8: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}%)')
print('=' * 80)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports (Paper Standard)

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
_sports_art = STAIR_ICAART_CONFIGS['sports']['artifact_dir']
_telemetry_sports = parse_telemetry_jsonl(_sports_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_sports:
    _eps = [r['epoch'] for r in _telemetry_sports]
    _vram_mb = [r.get('peak_allocated_gib', 0) * 1024 if 'peak_allocated_gib' in r else r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_sports]
    _max_vram = max(_vram_mb) if _vram_mb else 173.54
    plt.plot(_eps, _vram_mb, color='#ff7f0e', lw=2.2, label=f'Peak Allocated Tensor VRAM ({_max_vram:.2f} MiB)')
    plt.axhline(240, color='red', ls='--', lw=1.5, label='Theoretical Ceiling Budget (240 MiB)')
    plt.ylim(0, 260)
    plt.title('Amazon Sports — Peak Model Tensor VRAM Profile (STAIR-REEC / versionICAART)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=11)
    plt.ylabel('VRAM (MiB)', fontsize=11)
    plt.legend(loc='lower right', frameon=True)
    plt.tight_layout()
    _fig_path = str(WORK_DIR / 'reports' / 'vram_profile_sports.png')
    plt.savefig(_fig_path, dpi=300)
    plt.show()
    print(f'✅ Biểu đồ tiêu thụ VRAM Amazon Sports đã lưu tại: {_fig_path}')
else:
    print('⚠️ Chưa tìm thấy telemetry để vẽ biểu đồ VRAM Amazon Sports.')


## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports
_fig, _ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)
_sports_art = STAIR_ICAART_CONFIGS['sports']['artifact_dir']
_sports_log = STAIR_ICAART_CONFIGS['sports']['log']
_telemetry_sports = parse_telemetry_jsonl(_sports_art)
_bpr_sp, _cl_sp, _tot_sp = parse_training_losses(_sports_log, _telemetry_sports)
_val_ndcg_sp = parse_valid_metric(_sports_log, 'NDCG@20')

if _tot_sp:
    _eps_tot = [x[0] for x in _tot_sp]
    _vals_tot = [x[1] for x in _tot_sp]
    _ax1.plot(_eps_tot, _vals_tot, color='#d62728', lw=1.8, label='Total Loss')
if _bpr_sp:
    _eps_bpr = [x[0] for x in _bpr_sp]
    _vals_bpr = [x[1] for x in _bpr_sp]
    _ax1.plot(_eps_bpr, _vals_bpr, color='#1f77b4', lw=1.5, ls='--', label='BPR Loss')

_ax1.set_xlabel('Epoch', fontsize=11)
_ax1.set_ylabel('Loss Value', fontsize=11, color='#d62728')
_ax1.tick_params(axis='y', labelcolor='#d62728')

if _val_ndcg_sp:
    _ax2 = _ax1.twinx()
    _eps_val = [x[0] for x in _val_ndcg_sp]
    _vals_val = [x[1] for x in _val_ndcg_sp]
    _ax2.plot(_eps_val, _vals_val, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', fontsize=11, color='#2ca02c')
    _ax2.tick_params(axis='y', labelcolor='#2ca02c')

plt.title('Amazon Sports — Learning Dynamics & Convergence Profiles (STAIR-REEC)', fontsize=12, fontweight='bold')
_fig.tight_layout()
_curve_path = str(WORK_DIR / 'reports' / 'learning_curve_sports.png')
plt.savefig(_curve_path, dpi=300)
plt.show()
print(f'✅ Đường cong học tập Amazon Sports đã lưu tại: {_curve_path}')


## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn STAIR-ICAART)

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 7a: Training STAIR-REEC on Amazon Baby (Pha B: Confirmatory)
baby_time = run_training_stair_icaart('baby', delta=1.0, mode='replicated_excess')
ep_baby, metrics_baby = extract_test_metrics(STAIR_ICAART_CONFIGS['baby']['log'], STAIR_ICAART_CONFIGS['baby']['artifact_dir'])
val_ep_baby, val_m_baby = extract_best_validation(STAIR_ICAART_CONFIGS['baby']['log'])
diag_baby = parse_icaart_diagnostics(STAIR_ICAART_CONFIGS['baby']['log'])

STAIR_ICAART_RESULTS['baby'] = {
    'metrics': metrics_baby,
    'best_epoch': ep_baby,
    'val_metrics': val_m_baby,
    'train_time': baby_time,
    'diagnostics': diag_baby,
}

print('=' * 80)
print(f'🏆 KẾT QUẢ NGHIỆM THU AMAZON BABY (STAIR-REEC / Best Epoch: {ep_baby}):')
for m, v in metrics_baby.items():
    v8_val = V8_64_REF['baby'].get(m, 0.0)
    bl_val = BASELINE_REF['baby'].get(m, 0.0)
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    delta_bl = ((v - bl_val) / bl_val) * 100 if bl_val else 0.0
    print(f'  * {m:12s}: {v:.4f} (Baseline: {bl_val:.4f} | Δ vs BL: {delta_bl:+.2f}% | v8: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}%)')
print('=' * 80)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby (Paper Standard)

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
_baby_art = STAIR_ICAART_CONFIGS['baby']['artifact_dir']
_telemetry_baby = parse_telemetry_jsonl(_baby_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_baby:
    _eps = [r['epoch'] for r in _telemetry_baby]
    _vram_mb = [r.get('peak_allocated_gib', 0) * 1024 if 'peak_allocated_gib' in r else r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_baby]
    _max_vram_b = max(_vram_mb) if _vram_mb else 165.0
    plt.plot(_eps, _vram_mb, color='#1f77b4', lw=2.2, label=f'Peak Allocated Tensor VRAM ({_max_vram_b:.2f} MiB)')
    plt.axhline(220, color='red', ls='--', lw=1.5, label='Theoretical Ceiling Budget (220 MiB)')
    plt.ylim(0, 240)
    plt.title('Amazon Baby — Peak Model Tensor VRAM Profile (STAIR-REEC / versionICAART)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=11)
    plt.ylabel('VRAM (MiB)', fontsize=11)
    plt.legend(loc='lower right', frameon=True)
    plt.tight_layout()
    _fig_path_b = str(WORK_DIR / 'reports' / 'vram_profile_baby.png')
    plt.savefig(_fig_path_b, dpi=300)
    plt.show()
    print(f'✅ Biểu đồ tiêu thụ VRAM Amazon Baby đã lưu tại: {_fig_path_b}')
else:
    print('⚠️ Chưa tìm thấy telemetry để vẽ biểu đồ VRAM Amazon Baby.')


## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby
_fig, _ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)
_baby_art = STAIR_ICAART_CONFIGS['baby']['artifact_dir']
_baby_log = STAIR_ICAART_CONFIGS['baby']['log']
_telemetry_baby = parse_telemetry_jsonl(_baby_art)
_bpr_bb, _cl_bb, _tot_bb = parse_training_losses(_baby_log, _telemetry_baby)
_val_ndcg_bb = parse_valid_metric(_baby_log, 'NDCG@20')

if _tot_bb:
    _eps_tot = [x[0] for x in _tot_bb]
    _vals_tot = [x[1] for x in _tot_bb]
    _ax1.plot(_eps_tot, _vals_tot, color='#d62728', lw=1.8, label='Total Loss')
if _bpr_bb:
    _eps_bpr = [x[0] for x in _bpr_bb]
    _vals_bpr = [x[1] for x in _bpr_bb]
    _ax1.plot(_eps_bpr, _vals_bpr, color='#1f77b4', lw=1.5, ls='--', label='BPR Loss')

_ax1.set_xlabel('Epoch', fontsize=11)
_ax1.set_ylabel('Loss Value', fontsize=11, color='#d62728')
_ax1.tick_params(axis='y', labelcolor='#d62728')

if _val_ndcg_bb:
    _ax2 = _ax1.twinx()
    _eps_val = [x[0] for x in _val_ndcg_bb]
    _vals_val = [x[1] for x in _val_ndcg_bb]
    _ax2.plot(_eps_val, _vals_val, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', fontsize=11, color='#2ca02c')
    _ax2.tick_params(axis='y', labelcolor='#2ca02c')

plt.title('Amazon Baby — Learning Dynamics & Convergence Profiles (STAIR-REEC)', fontsize=12, fontweight='bold')
_fig.tight_layout()
_curve_path_b = str(WORK_DIR / 'reports' / 'learning_curve_baby.png')
plt.savefig(_curve_path_b, dpi=300)
plt.show()
print(f'✅ Đường cong học tập Amazon Baby đã lưu tại: {_curve_path_b}')


## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (Scale-up STAIR-ICAART)

In [ ]:
# Cell 8a: Training STAIR-ICAART on Amazon Electronics (Pha C: Scale-up)
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

# Tuỳ chọn chạy Amazon Electronics (Mặc định False để ưu tiên chu kỳ Pilot; đặt True khi đã phân bổ đủ quota 10-12 giờ)
RUN_ELECTRONICS = False

if not RUN_ELECTRONICS:
    print('ℹ️ Amazon Electronics đang ở chế độ chờ (RUN_ELECTRONICS = False).')
    print('   ↳ Đặt RUN_ELECTRONICS = True trong cell này khi bạn muốn khởi chạy huấn luyện quy mô lớn (~1.7M tương tác).')
else:
    elec_time = run_training_stair_icaart('electronics')
    ep_elec, metrics_elec = extract_test_metrics(STAIR_ICAART_CONFIGS['electronics']['log'], STAIR_ICAART_CONFIGS['electronics']['artifact_dir'])
    val_ep_elec, val_m_elec = extract_best_validation(STAIR_ICAART_CONFIGS['electronics']['log'])
    STAIR_ICAART_RESULTS['electronics'] = {
        'metrics': metrics_elec,
        'best_epoch': ep_elec,
        'val_metrics': val_m_elec,
        'train_time': elec_time,
    }

    print('=' * 80)
    print(f'🏆 KẾT QUẢ NGHIỆM THU AMAZON ELECTRONICS (STAIR-ICAART / Best Epoch: {ep_elec}):')
    for m, v in metrics_elec.items():
        v8_val = V8_64_REF['electronics'].get(m, 0.0)
        v1_val = STAIR6_V1_REF['electronics'].get(m, 0.0)
        delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
        delta_v1 = ((v - v1_val) / v1_val) * 100 if v1_val else 0.0
        print(f'  * {m:12s}: {v:.4f} (v8: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}% | v1: {v1_val:.4f} | Δ vs v1: {delta_v1:+.2f}%)')
    print('=' * 80)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics (Hardware Gate Validation)

In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Scale-up Hardware Gate Validation)
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

import matplotlib.pyplot as plt

_elec_art = STAIR_ICAART_CONFIGS['electronics']['artifact_dir']
_telemetry_elec = parse_telemetry_jsonl(_elec_art)

if _telemetry_elec or ('electronics' in vram_profile and len(vram_profile['electronics']) > 0):
    plt.figure(figsize=(9, 4.5), dpi=150)
    if _telemetry_elec:
        _eps = [r['epoch'] for r in _telemetry_elec]
        _vram_mb = [r.get('peak_allocated_gib', 0) * 1024 if 'peak_allocated_gib' in r else r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_elec]
        _max_vram_e = max(_vram_mb) if _vram_mb else 740.0
        plt.plot(_eps, _vram_mb, color='#2ca02c', lw=2.2, label=f'Peak Allocated Tensor VRAM ({_max_vram_e:.2f} MiB)')
        plt.axhline(800, color='red', ls='--', lw=1.5, label='Hardware Ceiling Safety Gate (800 MiB)')
        plt.ylim(0, 900)
        plt.title('Amazon Electronics — Peak Model Tensor VRAM Profile (STAIR-ICAART)', fontsize=12, fontweight='bold')
        plt.xlabel('Epoch', fontsize=10)
        plt.ylabel('Allocated VRAM (MiB)', fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='center right', frameon=True)
    elif 'electronics' in vram_profile and len(vram_profile['electronics']) > 0:
        plt.plot(vram_profile['electronics'], color='#2ca02c', lw=1.8, label='Host NVML GPU Used (MB)')
        plt.title('Amazon Electronics — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
        plt.xlabel('Seconds', fontsize=10)
        plt.ylabel('Memory (MB)', fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend()
    _p_vram_e = WORK_DIR / 'reports' / 'vram_profile_electronics.png'
    plt.savefig(_p_vram_e, bbox_inches='tight')
    plt.show()
    print(f'✅ Đã lưu biểu đồ VRAM Profile Electronics tại: {_p_vram_e}')
else:
    print('ℹ️ Chưa có dữ liệu telemetry VRAM cho Amazon Electronics.')


## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics

In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

import matplotlib.pyplot as plt

_elec_art = STAIR_ICAART_CONFIGS['electronics']['artifact_dir']
_elec_log = STAIR_ICAART_CONFIGS['electronics']['log']
_telemetry_elec = parse_telemetry_jsonl(_elec_art)
_bpr_el, _cl_el, _tot_el = parse_training_losses(_elec_log, _telemetry_elec)
_val_ndcg_el = parse_valid_metric(_elec_log, 'NDCG@20')

if _tot_el or _val_ndcg_el:
    _fig, _ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)
    if _tot_el:
        _eps_tot = [x[0] for x in _tot_el]
        _vals_tot = [x[1] for x in _tot_el]
        _ax1.plot(_eps_tot, _vals_tot, color='#d62728', lw=1.8, label='Total Loss')
    if _bpr_el:
        _eps_bpr = [x[0] for x in _bpr_el]
        _vals_bpr = [x[1] for x in _bpr_el]
        _ax1.plot(_eps_bpr, _vals_bpr, color='#1f77b4', lw=1.5, ls='--', label='BPR Loss')

    _ax1.set_xlabel('Epoch', fontsize=10)
    _ax1.set_ylabel('Training Loss', fontsize=10, color='#d62728')
    _ax1.tick_params(axis='y', labelcolor='#d62728')
    _ax1.grid(True, linestyle='--', alpha=0.4)

    if _val_ndcg_el:
        _ax2 = _ax1.twinx()
        _eps_ndcg = [x[0] for x in _val_ndcg_el]
        _vals_ndcg = [x[1] for x in _val_ndcg_el]
        _ax2.plot(_eps_ndcg, _vals_ndcg, color='#2ca02c', lw=2.0, marker='s', markersize=3, label='Val NDCG@20')
        _ax2.set_ylabel('Validation NDCG@20', fontsize=10, color='#2ca02c')
        _ax2.tick_params(axis='y', labelcolor='#2ca02c')

    plt.title('Amazon Electronics — Quá Trình Hội Tụ & Động Lực Học Hàm Mất Mát (STAIR-ICAART)', fontsize=11, fontweight='bold')
    _p_loss_e = WORK_DIR / 'reports' / 'learning_curve_electronics.png'
    plt.savefig(_p_loss_e, bbox_inches='tight')
    plt.show()
    print(f'✅ Đã lưu đồ thị Learning Curve Electronics tại: {_p_loss_e}')
else:
    print('ℹ️ Chưa có dữ liệu loss cho Amazon Electronics.')


## Cell 9a 📊 Bảng Đối Soát Hiệu Năng Đa Thế Hệ Toàn Diện (Benchmark Summary)

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 9a: Bảng Tổng Hợp Đối Soát Hiệu Năng Đa Thế Hệ (Benchmark Summary)
import os, sys
import pandas as pd
try:
    from prettytable import PrettyTable
except ImportError:
    class PrettyTable:
        def __init__(self):
            self.field_names = []
            self.rows = []
        def add_row(self, row):
            self.rows.append(row)
        def __str__(self):
            lines = [' | '.join(map(str, self.field_names))]
            lines.append('-' * len(lines[0]))
            for r in self.rows:
                lines.append(' | '.join(map(str, r)))
            return '\n'.join(lines)

if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}

_table = PrettyTable()
_table.field_names = ['Dataset', 'Kiến trúc / Phiên bản', *TRACKED_METRICS, 'Δ vs Baseline (%)', 'Δ vs v8-64D (%)', 'Δ vs v2 (%)']

_df_rows = []

for _ds in ('sports', 'baby', 'electronics'):
    _ref_bl = BASELINE_REF[_ds]['NDCG@20']
    _ref_v4 = V4_REF[_ds]['NDCG@20']
    _ref_v8 = V8_64_REF[_ds]['NDCG@20']
    _ref_v1 = STAIR6_V1_REF[_ds]['NDCG@20']
    _ref_v2 = STAIR6_V2_REF[_ds]['NDCG@20']

    # 1. Baseline
    _table.add_row([_ds, 'STAIR (baseline)', *[f"{BASELINE_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], '-', '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR (baseline)', **{_m: BASELINE_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': 0.0, 'Δ vs v8': 0.0, 'Δ vs v2': 0.0})

    # 2. v4
    _d_v4_bl = 100 * (_ref_v4 - _ref_bl) / _ref_bl
    _table.add_row([_ds, 'STAIR5-v4 (NLGCL-CSE)', *[f"{V4_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v4_bl:+.2f}%", '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v4', **{_m: V4_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v4_bl, 'Δ vs v8': 0.0, 'Δ vs v2': 0.0})

    # 3. v8-64D
    _d_v8_bl = 100 * (_ref_v8 - _ref_bl) / _ref_bl
    _table.add_row([_ds, 'STAIR5-v8 (WMSG-CSE)', *[f"{V8_64_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v8_bl:+.2f}%", '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v8 (WMSG-CSE)', **{_m: V8_64_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v8_bl, 'Δ vs v8': 0.0, 'Δ vs v2': 0.0})

    # 4. STAIR6-v1
    _d_v1_bl = 100 * (_ref_v1 - _ref_bl) / _ref_bl
    _d_v1_v8 = 100 * (_ref_v1 - _ref_v8) / _ref_v8
    _table.add_row([_ds, 'STAIR6-v1 (LCAE-WMSG-CSE)', *[f"{STAIR6_V1_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v1_bl:+.2f}%", f"{_d_v1_v8:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR6-v1 (LCAE-WMSG-CSE)', **{_m: STAIR6_V1_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v1_bl, 'Δ vs v8': _d_v1_v8, 'Δ vs v2': 0.0})

    # 5. STAIR6-v2
    _d_v2_bl = 100 * (_ref_v2 - _ref_bl) / _ref_bl
    _d_v2_v8 = 100 * (_ref_v2 - _ref_v8) / _ref_v8
    _table.add_row([_ds, 'STAIR6-v2 (AKP-WMSG-CSE)', *[f"{STAIR6_V2_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v2_bl:+.2f}%", f"{_d_v2_v8:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR6-v2 (AKP-WMSG-CSE)', **{_m: STAIR6_V2_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v2_bl, 'Δ vs v8': _d_v2_v8, 'Δ vs v2': 0.0})

    # 6. STAIR-REEC (versionICAART)
    if _ds in STAIR_ICAART_RESULTS and STAIR_ICAART_RESULTS[_ds].get('metrics'):
        _reec_m = STAIR_ICAART_RESULTS[_ds]['metrics']
        _ndcg_reec = _reec_m.get('NDCG@20', 0.0)
        _d_reec_bl = 100 * (_ndcg_reec - _ref_bl) / _ref_bl
        _d_reec_v8 = 100 * (_ndcg_reec - _ref_v8) / _ref_v8
        _d_reec_v2 = 100 * (_ndcg_reec - _ref_v2) / _ref_v2
        _table.add_row([
            _ds, 'STAIR-REEC (versionICAART)',
            *[f"{_reec_m.get(_m, 0.0):.4f}" for _m in TRACKED_METRICS],
            f"{_d_reec_bl:+.2f}%", f"{_d_reec_v8:+.2f}%", f"{_d_reec_v2:+.2f}%"
        ])
        _df_rows.append({'Dataset': _ds, 'Model': 'STAIR-REEC (versionICAART)', **{_m: _reec_m.get(_m, 0.0) for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_reec_bl, 'Δ vs v8': _d_reec_v8, 'Δ vs v2': _d_reec_v2})

print('=' * 120)
print('📊 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG ĐA THẾ HỆ (Baseline vs v4 vs v8 vs v1 vs v2 vs STAIR-REEC):')
print('=' * 120)
print(_table)

# Xuất ra file CSV
_df_summary = pd.DataFrame(_df_rows)
os.makedirs(str(WORK_DIR / 'reports'), exist_ok=True)
_summary_csv = str(WORK_DIR / 'reports' / 'stair_icaart_benchmark_summary.csv')
_df_summary.to_csv(_summary_csv, index=False)
print(f'\n✅ Báo cáo tổng hợp đã lưu tại: {_summary_csv}')


## Cell 9b 📐 Kiểm định Thống kê & Ý nghĩa Thống kê Đầy đủ (p-value, Wilcoxon, Paired t-test, Cohen's d, CI 95%)

In [ ]:
# Cell 9b: Kiểm định Ý nghĩa Thống kê Đầy đủ (p-value, Paired t-test, Wilcoxon, Bootstrap 95% CI & LaTeX Generator)
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

import os, sys, json
import numpy as np
import scipy.stats as stats
import pandas as pd
from prettytable import PrettyTable

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}

def compute_bootstrap_ci(proposed, comparator, n_bootstrap=10000, ci=0.95, seed=42):
    """Tính khoảng tin cậy Bootstrap 95% cho độ chênh lệch delta = proposed - comparator."""
    rng = np.random.RandomState(seed)
    diff = np.array(proposed) - np.array(comparator)
    boot_diffs = []
    n = len(diff)
    for _ in range(n_bootstrap):
        boot_sample = rng.choice(diff, size=n, replace=True)
        boot_diffs.append(np.mean(boot_sample))
    lower = np.percentile(boot_diffs, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_diffs, (1 + ci) / 2 * 100)
    return lower, upper

def compute_cohens_d(x, y):
    """Tính kích thước hiệu ứng Cohen's d cho mẫu ghép đôi."""
    diff = np.array(x) - np.array(y)
    sd = np.std(diff, ddof=1)
    if sd == 0:
        return 0.0
    return float(np.mean(diff) / sd)

def format_p_value(p):
    """Định dạng p-value và trả về ký hiệu ý nghĩa thống kê cho bài báo."""
    if p < 0.001:
        return f"{p:.2e}", "***"
    elif p < 0.01:
        return f"{p:.4f}", "**"
    elif p < 0.05:
        return f"{p:.4f}", "*"
    else:
        return f"{p:.4f}", "ns"

print('=' * 120)
print('📐 TIẾN HÀNH KIỂM ĐỊNH GIẢ THUYẾT THỐNG KÊ TOÀN DIỆN CHO VIẾT BÀI BÁO KHOA HỌC (PAPER SUBMISSION)...')
print('=' * 120)

stat_results = []
stat_table = PrettyTable()
stat_table.field_names = [
    'Tập dữ liệu', 'Chỉ số', 'STAIR-ICAART', 'V8-64D', 'Chênh lệch (Δ)', 
    'Δ (%)', 't-stat', 'p-value (t-test)', 'p-value (Wilcoxon)', 'Bootstrap 95% CI', "Cohen's d", 'Sig.'
]

for ds in ('sports', 'baby', 'electronics'):
    art_dir = STAIR_ICAART_CONFIGS[ds]['artifact_dir']
    log_file = STAIR_ICAART_CONFIGS[ds]['log']

    # 1. Đọc kết quả thực nghiệm unrounded từ selected_metrics.json nếu có
    selected_json = Path(art_dir) / 'selected_metrics.json'
    reec_metrics = {}
    if selected_json.exists():
        try:
            with open(selected_json, 'r', encoding='utf-8') as f:
                sj = json.load(f)
                reec_metrics = sj.get('metrics', {})
        except Exception:
            pass
    if not reec_metrics:
        reec_metrics = STAIR_ICAART_RESULTS.get(ds, {}).get('metrics', {})

    v8_metrics = V8_64_REF[ds]
    bl_metrics = BASELINE_REF[ds]

    # Kiểm tra xem tập dữ liệu đã có kết quả thực nghiệm chưa
    has_run = any(reec_metrics.get(m, 0.0) > 0 for m in TRACKED_METRICS)

    if not has_run:
        for metric in TRACKED_METRICS:
            stat_table.add_row([
                ds, metric, "NOT RUN", f"{v8_metrics[metric]:.4f}", "-",
                "-", "-", "-", "-", "-", "-", "pending"
            ])
        continue

    # Đọc chuỗi đánh giá validation từ evaluation_metrics.jsonl hoặc log để tính kiểm định thực tế
    val_jsonl = Path(art_dir) / 'evaluation_metrics.jsonl'
    val_series_prop = []
    if val_jsonl.exists():
        try:
            with open(val_jsonl, 'r', encoding='utf-8') as f:
                for line in f:
                    rec = json.loads(line)
                    if rec.get('mode') == 'valid' and 'metrics' in rec:
                        val_series_prop.append(rec['metrics'])
        except Exception:
            pass

    for metric in TRACKED_METRICS:
        val_prop = reec_metrics.get(metric, 0.0)
        val_v8 = v8_metrics.get(metric, 0.0)

        # Lấy các điểm validation trong cửa sổ hội tụ cuối (10 điểm gần nhất) nếu có
        metric_key = metric.upper()
        prop_window = [step[metric_key] for step in val_series_prop[-10:] if metric_key in step] if val_series_prop else []

        if len(prop_window) >= 5:
            # So sánh trên cửa sổ hội tụ thực tế
            v8_ref_series = np.full(len(prop_window), val_v8)
            sample_prop = np.array(prop_window)
            sample_v8 = v8_ref_series
            t_stat, p_val_ttest = stats.ttest_1samp(sample_prop, popmean=val_v8)
            try:
                w_stat, p_val_wilcoxon = stats.wilcoxon(sample_prop - val_v8)
            except Exception:
                p_val_wilcoxon = p_val_ttest
            ci_low, ci_high = compute_bootstrap_ci(sample_prop, sample_v8)
            d_val = compute_cohens_d(sample_prop, sample_v8)
        else:
            # Mẫu đơn lẻ (Single matched seed): Báo cáo chênh lệch chính xác không tạo mẫu giả
            t_stat = 0.0
            p_val_ttest = 1.0
            p_val_wilcoxon = 1.0
            ci_low = val_prop - val_v8
            ci_high = val_prop - val_v8
            d_val = 0.0

        delta_abs = val_prop - val_v8
        delta_pct = (delta_abs / val_v8) * 100 if val_v8 else 0.0
        p_str, sig_str = format_p_value(p_val_ttest)
        p_w_str, _ = format_p_value(p_val_wilcoxon)

        stat_table.add_row([
            ds, metric, f"{val_prop:.4f}", f"{val_v8:.4f}", f"{delta_abs:+.4f}",
            f"{delta_pct:+.2f}%", f"{t_stat:.2f}" if t_stat != 0.0 else "-",
            p_str if t_stat != 0.0 else "-",
            p_w_str if t_stat != 0.0 else "-",
            f"[{ci_low:.5f}, {ci_high:.5f}]", f"{d_val:.2f}" if d_val != 0.0 else "-", sig_str
        ])

        stat_results.append({
            'Dataset': ds,
            'Metric': metric,
            'STAIR_ICAART': val_prop,
            'STAIR5_v8': val_v8,
            'STAIR_Baseline': bl_metrics.get(metric, 0.0),
            'Delta_vs_v8': delta_abs,
            'Delta_vs_v8_pct': delta_pct,
            't_stat': t_stat,
            'p_value_ttest': p_val_ttest,
            'p_value_wilcoxon': p_val_wilcoxon,
            'ci_95_low': ci_low,
            'ci_95_high': ci_high,
            'cohens_d': d_val,
            'significance': sig_str
        })

print(stat_table)

# Xuất ra file CSV kiểm định thống kê
_p_csv_stat = WORK_DIR / 'reports' / 'stair_icaart_statistical_significance.csv'
pd.DataFrame(stat_results).to_csv(_p_csv_stat, index=False)
print(f'\n✅ Đã lưu kết quả kiểm định thống kê chính thức tại: {_p_csv_stat}')


## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints

In [ ]:
# Guard: Đảm bảo biến môi trường REPO_ROOT, WORK_DIR, STAIR_DIR, ENV và Reference Tables luôn sẵn sàng
if 'REPO_ROOT' not in globals() or 'WORK_DIR' not in globals() or 'ENV' not in globals():
    import os, sys
    from pathlib import Path
    _kw = '/kaggle/working'
    _safe_home = os.path.expanduser('~')
    _cands = [
        os.path.join(_kw, 'STAIR-Enhanced'),
        os.path.join(_safe_home, 'STAIR-Enhanced'),
        os.path.abspath('STAIR-Enhanced'),
        os.path.abspath('.'),
        _kw,
    ]
    _found_root = next((p for p in _cands if os.path.exists(os.path.join(p, 'models', 'stair_icaart.py')) or os.path.exists(os.path.join(p, 'main_stair_icaart.py'))), os.path.abspath('.'))
    REPO_ROOT = Path(_found_root).resolve()
    WORK_DIR = Path(_kw).resolve() if os.path.exists(_kw) else Path.cwd().resolve()
    STAIR_DIR = REPO_ROOT
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home, ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    for _p in [str(REPO_ROOT), str(WORK_DIR), _kw, _safe_home]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.insert(0, _p)

if 'TRACKED_METRICS' not in globals():
    TRACKED_METRICS = ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

if 'BASELINE_REF' not in globals():
    BASELINE_REF = {
        'sports':      {'Recall@1': 0.0142, 'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'baby':        {'Recall@1': 0.0113, 'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'electronics': {'Recall@1': 0.0091, 'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
    }
if 'V4_REF' not in globals():
    V4_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
        'baby':        {'Recall@1': 0.0126, 'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
        'electronics': {'Recall@1': 0.0098, 'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
    }
if 'V8_64_REF' not in globals():
    V8_64_REF = {
        'sports':      {'Recall@1': 0.0151, 'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0128, 'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0100, 'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V1_REF' not in globals():
    STAIR6_V1_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0760, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0519},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0461, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR6_V2_REF' not in globals():
    STAIR6_V2_REF = {
        'sports':      {'Recall@1': 0.0152, 'Recall@10': 0.0759, 'Recall@20': 0.1141, 'NDCG@10': 0.0419, 'NDCG@20': 0.0518},
        'baby':        {'Recall@1': 0.0129, 'Recall@10': 0.0674, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0463},
        'electronics': {'Recall@1': 0.0102, 'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
    }
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Telemetry, Reports)
import os, shutil, json, time, subprocess

if 'RUN_ID' not in globals():
    RUN_ID = 'default_run'
if 'STAIR_ICAART_CONFIGS' not in globals():
    STAIR_ICAART_CONFIGS = {}
if 'STAIR_ICAART_RESULTS' not in globals():
    STAIR_ICAART_RESULTS = {}

_archive_dir = str(WORK_DIR / 'stair_icaart_artifacts')
os.makedirs(_archive_dir, exist_ok=True)

# 1. Thu thập logs, reports và runs
_target_dirs = [
    str(WORK_DIR / 'logs' / 'stair_icaart'),
    str(WORK_DIR / 'reports'),
    str(WORK_DIR / 'runs' / 'stair_icaart'),
    os.path.expanduser('~/logs/stair_icaart'),
    os.path.expanduser('~/reports'),
]
for _src in _target_dirs:
    if os.path.exists(_src):
        _dst = os.path.join(_archive_dir, os.path.basename(_src))
        try:
            shutil.copytree(_src, _dst, dirs_exist_ok=True)
        except Exception:
            pass

# 2. Tạo manifest tổng thể cho phiên thử nghiệm
_manifest_data = {
    'run_id': RUN_ID,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'framework': 'STAIR-REEC (versionICAART)',
    'primary_delta': 1.0,
    'primary_tau': 2.0,
    'results': {k: v.get('metrics', {}) for k, v in STAIR_ICAART_RESULTS.items()},
    'configs': {k: {cfg_k: str(cfg_v) for cfg_k, cfg_v in v.items()} for k, v in STAIR_ICAART_CONFIGS.items()},
}
with open(os.path.join(_archive_dir, 'run_manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(_manifest_data, f, indent=2, ensure_ascii=False)

# 3. Tạo file nén tar.gz và zip trong /kaggle/working
_tar_path = str(WORK_DIR / 'stair_icaart_artifacts.tar.gz')
_zip_path = str(WORK_DIR / 'stair_icaart_artifacts.zip')

print('📦 Đang nén toàn bộ artifacts STAIR-REEC...')
try:
    subprocess.run(['tar', '-czf', _tar_path, '-C', str(WORK_DIR), 'stair_icaart_artifacts'], check=True)
    print(f'✅ Đã tạo tệp tin nén Tarball: {_tar_path} ({os.path.getsize(_tar_path)/(1024**2):.2f} MB)')
except Exception as _e:
    print(f'⚠️ Tarball nén bỏ qua: {_e}')

try:
    shutil.make_archive(str(WORK_DIR / 'stair_icaart_artifacts'), 'zip', str(WORK_DIR), 'stair_icaart_artifacts')
    print(f'✅ Đã tạo tệp tin nén Zip: {_zip_path} ({os.path.getsize(_zip_path)/(1024**2):.2f} MB)')
except Exception as _e:
    print(f'⚠️ Zip nén bỏ qua: {_e}')

print('=' * 80)
print('🎉 TOÀN BỘ MINH CHỨNG VÀ KẾT QUẢ THỰC NGHIỆM STAIR-REEC ĐÃ ĐƯỢC ĐÓNG GÓI THÀNH CÔNG!')
print('=' * 80)
